
# Interactive Figure‑Eight Warping (Periodic $B_2^4$ Segments)

This notebook lets you interactively adjust **weights** for periodic Bernstein bumps that warp a
base figure‑eight (lemniscate) in $(\phi,\beta)$, and compare **original vs warped** curves.

**How to use**
1. Run all cells.
2. The last cell calls `build_app(4)` to create sliders for **4 bumps** per channel.
3. To use a different number (e.g. 10), change to `build_app(10)` and re‑run that cell.


In [12]:

import numpy as np
import matplotlib.pyplot as plt

def bernstein_i2_order4_periodic_segment(t, a, width=0.5, normalize=False):
    """Periodic B_2^4 'bump' on circular t ∈ [0,1). Nonzero on [a, a+width) modulo 1."""
    t = np.asarray(t, dtype=float) % 1.0
    width = float(width)
    if not (0.0 < width <= 1.0):
        raise ValueError("width must be in (0, 1].")
    delta = (t - a) % 1.0
    mask = delta <= width
    out = np.zeros_like(t)
    if np.any(mask):
        s = delta[mask] / width
        val = 6.0 * (s**2) * ((1.0 - s) ** 2)
        if normalize:
            val = val / width
        out[mask] = val
    return out

def progress_from_phase(t):
    th_phi  = np.mod(np.arctan2(np.sin(2*np.pi*t), np.cos(2*np.pi*t)), 2*np.pi)
    u_phi   = (th_phi / np.pi) % 1.0
    th_beta = np.mod(np.arctan2(np.sin(4*np.pi*t), np.cos(4*np.pi*t)), 2*np.pi)
    u_beta  = (th_beta / (0.5*np.pi)) % 1.0
    return u_phi, u_beta

def build_shape(u, K, width, weights):
    weights = np.asarray(weights, dtype=float)
    if len(weights) != K:
        raise ValueError(f"weights length {len(weights)} must equal K={K}")
    N = np.ones_like(u)
    for k, w in enumerate(weights):
        center = k / K
        start = center - width/2.0
        N += w * bernstein_i2_order4_periodic_segment(u, a=start, width=width)
    return N

def warp_class_with_shape(t, a_phi, b_beta, c_phi, c_beta,
                          w_phi, w_beta, width_phi=0.5, width_beta=0.5,
                          left_first=True):
    sgn = -1.0 if left_first else +1.0
    phi_c  = c_phi  + sgn * a_phi  * np.sin(2*np.pi*t)
    beta_c = c_beta +       b_beta * np.sin(4*np.pi*t)
    u_phi, u_beta = progress_from_phase(t)
    N_phi  = build_shape(t,  K=len(w_phi),  width=width_phi,  weights=w_phi)
    N_beta = build_shape(t, K=len(w_beta), width=width_beta, weights=w_beta)
    phi_w  = c_phi  + (phi_c  - c_phi)  * N_phi
    beta_w = c_beta + (beta_c - c_beta) * N_beta
    return phi_w, beta_w, phi_c, beta_c


In [13]:

def plot_original_vs_warped(
    a_phi_deg, b_beta_deg, c_phi_deg, c_beta_deg,
    width_phi, width_beta, left_first,
    K, w_phi_vals, w_beta_vals, n_samples=2000
):
    a_phi  = np.deg2rad(a_phi_deg)
    b_beta = np.deg2rad(b_beta_deg)
    c_phi  = np.deg2rad(c_phi_deg)
    c_beta = np.deg2rad(c_beta_deg)

    t = np.linspace(0, 1, n_samples, endpoint=False)

    phi_w, beta_w, phi_c, beta_c = warp_class_with_shape(
        t, a_phi, b_beta, c_phi, c_beta,
        w_phi=w_phi_vals, w_beta=w_beta_vals,
        width_phi=width_phi, width_beta=width_beta,
        left_first=left_first
    )

    plt.figure(figsize=(7.0, 5.6))
    plt.plot(np.degrees(phi_c), np.degrees(beta_c), linestyle='--', alpha=0.4, label='class')
    plt.plot(np.degrees(phi_w), np.degrees(beta_w),                label='warped')
    plt.xlabel('φ [deg]'); plt.ylabel('β [deg]')
    plt.title('(φ, β) plane — original vs. warped')
    plt.axis('equal'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(7.5, 4.8))
    plt.plot(t, np.degrees(phi_c),  linestyle='--', alpha=0.5, label='φ class')
    plt.plot(t, np.degrees(phi_w),                label='φ warped')
    plt.plot(t, np.degrees(beta_c), linestyle='--', alpha=0.5, label='β class')
    plt.plot(t, np.degrees(beta_w),               label='β warped')
    plt.xlabel('t'); plt.ylabel('angle [deg]')
    plt.title('Original vs. Warped — time series')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    


In [14]:

from ipywidgets import FloatSlider, IntSlider, Checkbox, VBox, HBox, Layout, interactive_output
from IPython.display import display

def build_app(K=4):
    if K < 1 or K > 16:
        raise ValueError("Choose 1 ≤ K ≤ 16 for a manageable UI.")

    a_phi_deg_s  = FloatSlider(value=22.0, min=0.0,  max=90.0, step=0.2, description='aφ [deg]', layout=Layout(width='45%'))
    b_beta_deg_s = FloatSlider(value= 9.0, min=0.0,  max=90.0, step=0.2, description='bβ [deg]', layout=Layout(width='45%'))
    c_phi_deg_s  = FloatSlider(value= 0.0, min=-90.0, max=90.0, step=0.2, description='cφ [deg]', layout=Layout(width='45%'))
    c_beta_deg_s = FloatSlider(value=35.0, min=  0.0, max= 85.0, step=0.2, description='cβ [deg]', layout=Layout(width='45%'))

    width_phi_s  = FloatSlider(value=0.5, min=0.05, max=1.0, step=0.05, description='width φ',  layout=Layout(width='45%'))
    width_beta_s = FloatSlider(value=0.5, min=0.05, max=1.0, step=0.05, description='width β', layout=Layout(width='45%'))

    left_first_s = Checkbox(value=True, description='left-first')

    w_phi_sl = [FloatSlider(value=0.00, min=-1.0, max=1.0, step=0.01, description=f'wφ[{i+1}]', layout=Layout(width='45%')) for i in range(K)]
    w_beta_sl= [FloatSlider(value=0.00, min=-1.0, max=1.0, step=0.01, description=f'wβ[{i+1}]', layout=Layout(width='45%')) for i in range(K)]

    controls = dict(
        a_phi_deg=a_phi_deg_s, b_beta_deg=b_beta_deg_s, c_phi_deg=c_phi_deg_s, c_beta_deg=c_beta_deg_s,
        width_phi=width_phi_s, width_beta=width_beta_s, left_first=left_first_s,
        K=IntSlider(value=K, min=K, max=K, step=1, description='K', layout=Layout(width='25%'))
    )
    for i, s in enumerate(w_phi_sl):
        controls[f'wphi_{i+1}'] = s
    for i, s in enumerate(w_beta_sl):
        controls[f'wbeta_{i+1}'] = s

    def _callback(a_phi_deg, b_beta_deg, c_phi_deg, c_beta_deg,
                  width_phi, width_beta, left_first, K, **kw):
        w_phi_vals  = np.array([kw[f'wphi_{i+1}']  for i in range(K)], dtype=float)
        w_beta_vals = np.array([kw[f'wbeta_{i+1}'] for i in range(K)], dtype=float)
        plot_original_vs_warped(
            a_phi_deg, b_beta_deg, c_phi_deg, c_beta_deg,
            width_phi, width_beta, left_first,
            K, w_phi_vals, w_beta_vals
        )

    out = interactive_output(_callback, controls)

    ui = VBox([
        HBox([a_phi_deg_s, b_beta_deg_s]),
        HBox([c_phi_deg_s,  c_beta_deg_s ]),
        HBox([width_phi_s,  width_beta_s, left_first_s]),
        VBox(w_phi_sl),
        VBox(w_beta_sl),
        out
    ])
    display(ui)


In [15]:

# Create sliders for K=4 bumps per channel.
build_app(10)
